# HW2 – 5. Query-based detector (RT-DETR)

RT-DETR-L (Ultralytics, COCO-pretrained) trained under the protocol from notebook 01 with the same budget per run as the detectors in notebooks 02–04. RT-DETR is a transformer detector: a fixed set of object queries is matched one-to-one to the ground truth, so no anchors and no NMS are needed.

**Experiments** (each starts from the best configuration so far):
1. **Separate learning rates** – one rate for the whole model vs a backbone rate 10× lower.
2. **Denoising loss** – 100 denoising queries (default) vs 200 and 400, because the default gives this dense dataset very few denoising groups (section 3).
3. **Test-time input resolution** on the best training (no retraining), and a check that NMS adds nothing.

We explain how bipartite matching removes NMS and changes training, plot validation mAP over training against YOLO11s under the same budget, and discuss small objects, dense scenes and convergence.

## 0. Setup

In [ ]:
# Make the repo importable: Kaggle clones it, locally the root is the parent of notebooks/
import os, sys
from pathlib import Path
if os.path.exists("/kaggle/working"):
    if not os.path.exists("/kaggle/working/repo"):
        !git clone --depth 1 https://github.com/BohdanZasymovych/ai-hw2-object-detection.git /kaggle/working/repo
    !pip install -q pycocotools huggingface_hub kaggle ultralytics==8.4.207   # only if missing; torch/torchvision stay Kaggle's own CUDA builds
    ROOT_DIR = Path("/kaggle/working/repo")
else:
    ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

In [ ]:
# Live reload of utils/common.py when editing locally (Kaggle's IPython autoreload is broken on Python 3.13)
if not os.path.exists("/kaggle/working"):
    %load_ext autoreload
    %autoreload 2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import ultralytics
from torchvision.ops import batched_nms
from utils.common import *
from utils.ultra import ULTRA_VERSION, YOLO_DATA_DIR, export_yolo, load_ultra, rtdetr_speed, train_runs, ultra_predictor

seed_everything()
print(env_info(), "| ultralytics:", ultralytics.__version__, "| data:", RAW_DIR)
if ultralytics.__version__ != ULTRA_VERSION:
    print(f"Warning: tested with ultralytics {ULTRA_VERSION}; the patches in utils/ultra.py depend on its internals")

Stages run in this session are listed in `RUN`; finished stages are downloaded from the Hugging Face repo. On Kaggle with **GPU T4 ×2** a Run All trains everything in about 3 h: the two trainings of each round run concurrently, one per GPU (one after another on a single GPU, about 5.2 h). Locally nothing is trained and the tables and plots are rebuilt from the saved runs. `QUICK=True` is a local smoke test on a few frames.

In [ ]:
QUICK = False                                    # True: local smoke test (few frames, 1 epoch, small input, nothing pushed)
ALL_STAGES = {"speed", "baseline", "lr", "nd200", "nd400", "final"}
RUN = ALL_STAGES if os.path.exists("/kaggle/working") or QUICK else set()   # stages computed in this session; the rest are pulled from the HF repo
SUBMIT = False        # True: write the test predictions of the tuned model and submit them to Kaggle
RETRAIN_FULL = False  # True: first retrain the tuned configuration on train + val and use these weights

# Fail early if trained stages could not be uploaded (needs HF_TOKEN with write access)
if RUN and not QUICK:
    check_hf_write()

## 1. Training setup

- **Model:** RT-DETR-L (HGNetv2 backbone, hybrid encoder on P3–P5, 6 decoder layers, 300 queries; about 32M parameters), COCO weights. Only the layers that depend on the number of classes are new (classification heads and denoising class embedding).
- **Data:** the same frames as notebooks 02–04 (every 3rd, 8,090 train / 951 val), resized once to a 1280 px long side (`export_yolo`), the same labels and Ultralytics' standard augmentation recipe as notebook 04 (mosaic, flip 0.5, HSV, translation, scale jitter ±25%).
- **Input:** RT-DETR in Ultralytics stretches each frame to a square, at training and at inference. **960×960 has the same number of pixels as the 1280×720 input of the other detectors** (objects keep their area, but width ×0.75 and height ×1.33), so it is the default. Section 4 can select 1280×1280 instead.
- **Optimisation:** AdamW, lr 1e-4, weight decay 1e-4 (RT-DETR's own recipe), batch 4, **nominal batch 16** (`nbs`; 4 accumulation steps). Ultralytics' default of 64 would give about 126 optimizer steps per epoch at this low rate, against about 505 here. Warm-up for half an epoch, with the bias rate starting at 0 instead of the default 0.1, which is 1000× the base rate here. Then linear decay to 1%, AMP, seed 42. Mosaic is off for the last epoch.
- **Budget:** 1.1 h of training per run (incl. validation), as in notebook 04. Ultralytics re-estimates the number of epochs after each epoch, so the LR schedule ends with the budget.
- **Reproducibility:** `deterministic=False`, because the `grid_sample` used by RT-DETR's decoder does not support deterministic mode (Ultralytics' note); the seed is still set, so repeated runs can differ slightly.
- **Evaluation:** the shared `coco_evaluate` on the val frames (ignore regions as `iscrowd`, up to 300 detections, score ≥ 0.001), on Ultralytics' best epoch; speed at batch size 1, FP16.

In [ ]:
CFG = dict(
    arch="rtdetr-l", weights="rtdetr-l.pt", long_side=640 if QUICK else PREP["long_side"],
    train_frames=16 if QUICK else None, val_frames=8 if QUICK else None, num_workers=4, seed=SEED,
    train=dict(  # Ultralytics train() arguments shared by every run; imgsz comes from the speed test (section 4)
        batch=2 if QUICK else 4, epochs=1 if QUICK else 100, time=None if QUICK else 1.1, optimizer="AdamW",
        lr0=1e-4, weight_decay=1e-4, nbs=16, warmup_epochs=0 if QUICK else 0.5, warmup_bias_lr=0.0,
        close_mosaic=0 if QUICK else 1, scale=0.25, workers=2 if QUICK else 4, seed=SEED, deterministic=False,
        max_det=EVAL_DETS, plots=False, cache=False),
)
DEFAULT_MODEL = dict(backbone_lr_ratio=1.0, nd=100)    # single learning rate, 100 denoising queries (Ultralytics default)
BACKBONE_LR_RATIO = 0.1                                # backbone 1e-5, rest 1e-4
ND = {"nd200": 200, "nd400": 400}                      # denoising queries of experiment 2
SPEED_SIZES = (320, 640) if QUICK else (960, 1280)     # candidate training inputs (default, larger)
MAX_COST_RATIO = 1.3                                   # the larger input is used only if it costs less than this
GRID = [320, 640] if QUICK else [800, 960, 1120, 1280, 1440]   # test-time inputs on the best training


def name(stage):
    return f"rtdetr_{stage}" + ("_quick" if QUICK else "")


print(CFG)

## 2. Data export and helpers

Same export as notebook 04: the kept frames are resized once and their label files copied next to them (`utils/ultra.py`). The frames, classes and resolution are the same as for `DetDataset`.

For evaluation the frames stay at a 1280 px long side and `imgsz` is passed separately: RT-DETR stretches them to the square itself, and resizing them to 960 px first would lose vertical detail (a 1280×720 frame would become 960×540 and be upsampled back).

In [ ]:
TRAIN_STAGES = {"baseline", "lr", "nd200", "nd400"}
DATA_YAML = None
if RUN & TRAIN_STAGES:
    DATA_YAML = export_yolo(YOLO_DATA_DIR / ("quick" if QUICK else str(CFG["long_side"])), long_side=CFG["long_side"],
                            max_frames=CFG["train_frames"])

In [ ]:
def ensure_run(stage, weights=False):
    # Local folder of a finished stage; downloaded from the HF repo if it was produced in another session
    return fetch_run(name(stage), weights, hint=f"Add '{stage}' to RUN to compute it in this session (RUN = {RUN}).")


def model_cfg(stage):
    return json.load(open(ensure_run(stage) / "config.json"))["model"]


def load_model(stage):
    return load_ultra(ensure_run(stage, weights=True) / "best.pt")


def frames_side(imgsz):
    # Long side of the frames given to the predictor: never below imgsz, otherwise the stretch would upsample them
    return max(CFG["long_side"], imgsz)


def evaluate(model, imgsz):
    return evaluate_model(ultra_predictor(model, imgsz), "val", max_frames=CFG["val_frames"], batch_size=1,
                          num_workers=CFG["num_workers"], label_offset=0, long_side=frames_side(imgsz))


def best_so_far(stages):
    # Finished stage with the highest val mAP
    done = {}
    for s in stages:
        try:
            done[s] = json.load(open(ensure_run(s) / "metrics.json"))["mAP"]
        except FileNotFoundError:
            pass
    return max(done, key=done.get)

In [ ]:
def train_stages(stages):
    # Train the given stages ({stage: model settings}) under the fixed protocol, concurrently on 2 GPUs if available,
    # then evaluate each best epoch with the shared COCO evaluation, save and upload the run
    cfgs = {s: dict(name=name(s), data=str(DATA_YAML), arch=CFG["arch"], weights=CFG["weights"],
                    **{**DEFAULT_MODEL, **m}, train={**CFG["train"], "imgsz": IMGSZ}) for s, m in stages.items()}
    train_runs(list(cfgs.values()))
    for s, c in cfgs.items():
        model = load_ultra(run_dir(c["name"]) / "best.pt")
        metrics = evaluate(model, IMGSZ)
        save_run(c["name"], config={**CFG, "model": {k: c[k] for k in DEFAULT_MODEL}, "test": dict(imgsz=IMGSZ)},
                 metrics=metrics)                               # history.json is written by the training process
        if not QUICK:
            push_run(c["name"])
        del model
        torch.cuda.empty_cache()


def row_name(stage):
    # Learning rates and denoising queries used in training + test-time input
    c = json.load(open(ensure_run(stage) / "config.json"))
    mc, t = c["model"], c["test"]
    lr = f"backbone lr x{mc['backbone_lr_ratio']:g}" if mc["backbone_lr_ratio"] != 1 else "single lr"
    return f"{'final: ' if stage == 'final' else ''}RT-DETR-L {lr}, nd {mc['nd']}, {t['imgsz']} px"


def show(stages):
    # Metrics of finished stages, one row per stage
    return pd.DataFrame({row_name(s): {k: v for k, v in json.load(open(ensure_run(s) / "metrics.json")).items() if k != "per_class"}
                         for s in stages}).T.round(3)

## 3. Bipartite matching, NMS and denoising

**How the matching works.** Each image gets a fixed set of 300 queries, chosen by the encoder as its top-scoring locations. During training the Hungarian algorithm finds the one-to-one assignment between queries and GT boxes with the lowest total cost, here $2\,\text{class} + 5\,L_1 + 2\,\text{GIoU}$. Each GT box gets exactly one query; all the others must predict "no object".

**Why NMS is not needed.** A second query on the same object is an unmatched query, so the loss pushes its score down. Decoder self-attention lets queries see each other, so they can learn which of them keeps an object. Duplicates are therefore suppressed by training rather than removed afterwards. Section 7 checks this by applying NMS to the predictions.

**How this changes training.**
- **Few positives:** one positive per GT box, against up to 10 for the task-aligned assigner of YOLO11 (notebook 04, section 3), so the loss has fewer positive examples per image, which slows convergence. Auxiliary losses on every decoder layer and on the encoder output add supervision.
- **Unstable targets:** a query's target can change between iterations as the predictions change, especially early in training.
- **Denoising:** noised copies of the GT boxes are given to the decoder as extra queries and must be reconstructed. Their target is known, so no matching is involved, which stabilises training. Each group holds positive and negative copies of a batch's boxes, so the **number of groups is `nd` divided by the largest number of boxes in an image of the batch**.

This dataset is much denser than COCO, so the default `nd=100` leaves very little room. Measured below by sampling batches of the training frames (without augmentation): the number of denoising groups per batch, the batches with a single group, and the batches where an image has more boxes than `nd`, so only a random subset of its boxes is denoised.

In [ ]:
def denoising_stats(nd, batch=CFG["train"]["batch"], n_batches=20_000):
    # Denoising groups per batch for `nd` denoising queries, from the number of boxes per kept train frame
    frames, boxes, _ = load_split("train")
    frames = subsample(frames)
    n = boxes.groupby("image_id").size().reindex(frames.image_id, fill_value=0).to_numpy()
    rng = np.random.default_rng(SEED)
    most = n[rng.integers(len(n), size=(n_batches, batch))].max(1)
    most = most[most > 0]                                   # batches without boxes have no denoising
    capped = np.minimum(most, nd)                           # Ultralytics: num_group = nd // min(largest image, nd)
    groups = nd // capped
    return {"groups per batch": groups.mean(), "% batches with one group": (groups == 1).mean() * 100,
            "% batches with an image above nd": (most > nd).mean() * 100, "denoising queries per image": (2 * groups * capped).mean()}


pd.DataFrame({f"nd = {nd}": denoising_stats(nd) for nd in [100, *ND.values()]}).T.round(1)

**Findings.** With the default `nd=100`, most batches have only one denoising group, so denoising is nearly switched off compared with COCO, where images have far fewer boxes. `nd=200` and `nd=400` restore several groups at the cost of more decoder queries in training (no extra weights, and no cost at inference), which is the reason for experiment 2.

## 4. Speed test: training input

The input size decides how many epochs fit the budget, so it is chosen first. This measures the training cost at 960 and 1280 px (synthetic images and 46 boxes per image, the dataset's mean, without data loading) and the memory of the most demanding setting of experiment 2 (`nd=400`, 140 boxes per image, about the 99th percentile), so a run cannot fail from lack of memory hours into the session.

**Rule:** 1280 px has 1.78× the pixels of 960 px, so it is used if it costs less than 1.3× as much per image and fits in memory; otherwise 960 px. A local test on an RTX 2050 measured 425 vs 451 ms/img, a much smaller gap than the pixel counts suggest, which is why this is checked on the T4 rather than assumed.

In [ ]:
if "speed" in RUN:
    low, high = SPEED_SIZES
    rows = [rtdetr_speed(s, batch=CFG["train"]["batch"], iters=8 if QUICK else 30) for s in SPEED_SIZES]
    speed = pd.DataFrame(rows).set_index("imgsz")
    worst_case = lambda s: rtdetr_speed(s, batch=CFG["train"]["batch"], nd=max(ND.values()), n_gt=140, iters=3)
    worst = worst_case(high)
    use_high = np.isfinite(worst["ms_per_img"]) and speed.ms_per_img[high] / speed.ms_per_img[low] < MAX_COST_RATIO
    if not use_high:
        worst = worst_case(low)
        if not (np.isfinite(worst["ms_per_img"]) and np.isfinite(speed.ms_per_img[low])):   # stop now, not hours into the session
            raise RuntimeError(f"Batch {CFG['train']['batch']} does not fit the GPU at {low} px: lower the batch size in CFG")
    d = save_run(name("speed"), config=dict(worst_case=worst, batch=CFG["train"]["batch"]), metrics=dict(imgsz=high if use_high else low))
    speed.to_csv(d / "speed.csv")
    if not QUICK:
        push_run(name("speed"))
d = ensure_run("speed")
IMGSZ = json.load(open(d / "metrics.json"))["imgsz"]
speed, worst = pd.read_csv(d / "speed.csv", index_col="imgsz"), json.load(open(d / "config.json"))["worst_case"]
speed["minutes per epoch (train only)"] = speed.ms_per_img * len(subsample(load_split("train")[0])) / 60_000
print(f"worst case (nd=400, 140 boxes per image, {worst['imgsz']} px): peak {worst['peak_gb']:.1f} GB, fits: {np.isfinite(worst['ms_per_img'])}")
print("selected training input:", IMGSZ)
speed.round(2)

**Findings.** *To fill in after the Kaggle run: measured cost per image, memory and the selected input.*

## 5. Baseline and experiment 1 – Separate learning rates

**Baseline:** one learning rate (1e-4) for the whole model, 300 queries, 100 denoising queries.

**Experiment 1:** backbone 1e-5, rest 1e-4, the ratio of RT-DETR's own recipe. The COCO backbone features already work, and the budget is short, so a lower rate should protect them while the new classification layers and the matching settle.

In [ ]:
train_stages({s: m for s, m in {"baseline": {}, "lr": {"backbone_lr_ratio": BACKBONE_LR_RATIO}}.items() if s in RUN})
show(["baseline", "lr"])

**Findings.** *To fill in after the Kaggle run: effect of the lower backbone rate on mAP, AP_small and the validation curve.*

## 6. Experiment 2 – Denoising loss

Starting from the better training of section 5, `nd` is raised to 200 and 400 (section 3). The denoising loss is a part of the training loss with a known target, so more groups mean more stable supervision per image, which matters most when only a few epochs fit the budget. The cost is more decoder queries during training, so slightly fewer epochs fit.

In [ ]:
if RUN & set(ND):
    best_lr = model_cfg(best_so_far(["baseline", "lr"]))
    train_stages({s: {**best_lr, "nd": nd} for s, nd in ND.items() if s in RUN})
show(["baseline", "lr", *ND])

**Findings.** *To fill in after the Kaggle run: search range 100 / 200 / 400, selected value, and whether more denoising helped within the budget.*

## 7. Tuned model: test-time input resolution and NMS

Final model ("tuned"): the best training by val mAP, evaluated at several square inputs (no retraining). Settings are selected on val and its predictions are saved for the analysis below.

In [ ]:
TRAINED = ["baseline", "lr", *ND]
if "final" in RUN:
    src = best_so_far(TRAINED)
    model = load_model(src)
    rows, best = [], None
    for sz in GRID:
        ds = DetDataset("val", label_offset=0, max_frames=CFG["val_frames"], long_side=frames_side(sz))
        p, ms = predict_dataset(ultra_predictor(model, sz), ds, batch_size=1, num_workers=CFG["num_workers"])
        m = {**coco_evaluate(p, "val", image_ids=ds.image_ids), "ms_per_img": ms}
        rows.append({"imgsz": sz, **{k: v for k, v in m.items() if k != "per_class"}})
        if best is None or m["mAP"] > best[1]["mAP"]:
            best = (dict(imgsz=sz), m, p)
    test, metrics, p = best
    d = save_run(name("final"), config={"trained": name(src), "model": model_cfg(src), "test": test, "image_ids": ds.image_ids},
                 metrics=metrics, preds=p)
    pd.DataFrame(rows).to_csv(d / "grid.csv", index=False)
    if not QUICK:
        push_run(name("final"))
    del model
    torch.cuda.empty_cache()
d = ensure_run("final")
final_cfg, final = json.load(open(d / "config.json")), json.load(open(d / "metrics.json"))
preds = json.load(open(d / "preds_val.json"))
print("trained:", final_cfg["trained"], "| best test-time setting:", final_cfg["test"])
pd.read_csv(d / "grid.csv").set_index("imgsz").round(3)

In [ ]:
# Search range and selected setting of every tuned parameter (each selected by val mAP; see the tables above)
sel = final_cfg["model"]
pd.DataFrame({
    "search range": {"backbone lr / rest (training)": f"1, {BACKBONE_LR_RATIO}", "denoising queries (training)": f"{DEFAULT_MODEL['nd']}, {', '.join(map(str, ND.values()))}",
                     "input, px (test)": ", ".join(map(str, GRID))},
    "selected": {"backbone lr / rest (training)": f"{sel['backbone_lr_ratio']:g}", "denoising queries (training)": str(sel["nd"]),
                 "input, px (test)": str(final_cfg["test"]["imgsz"])},
}).rename_axis("parameter")

**Does NMS add anything?** The tuned predictions with per-class NMS applied afterwards, at two IoU thresholds. No retraining or new inference is needed. The share of duplicate detections (score ≥ 0.3, same definition as `match_errors`) is compared with YOLO11s below.

In [ ]:
def apply_nms(results, iou):
    # Per-class NMS over COCO-format results (xywh, native pixels)
    if iou is None:
        return results
    r = pd.DataFrame(results)
    xywh = np.array(r.bbox.tolist(), dtype=np.float32).reshape(-1, 4)
    boxes = torch.from_numpy(np.c_[xywh[:, :2], xywh[:, :2] + xywh[:, 2:]])
    groups = torch.from_numpy(r.category_id.to_numpy() * 10**7 + r.image_id.to_numpy() % 10**7)   # one NMS per image and class
    keep = batched_nms(boxes, torch.from_numpy(r.score.to_numpy(np.float32)), groups, iou).tolist()
    return [results[i] for i in keep]


nms_rows = {}
for t in [None, 0.7, 0.5]:
    m = coco_evaluate(apply_nms(preds, t), "val", image_ids=final_cfg["image_ids"])
    nms_rows["no NMS" if t is None else f"NMS {t}"] = {k: v for k, v in m.items() if k != "per_class"}
pd.DataFrame(nms_rows).T.round(3)

**Findings.** *To fill in after the Kaggle run: selected input, how mAP and AP_small change with it, and whether NMS changes mAP.*

## 8. Comparison with the CNN detectors

Same val frames and evaluation for all models. Rows: tuned Faster R-CNN, RetinaNet and YOLO11s (notebooks 02–04), then the RT-DETR trainings and the tuned model. Speed is comparable only on the same GPU: the Faster R-CNN row was timed on a local RTX 2050 (about 76 ms/img on a T4).

In [ ]:
# Tuned models from notebooks 02-04 (metrics and predictions, no weights)
CNN = {"frcnn_final": "Faster R-CNN, tuned (notebook 02)", "retina_final": "RetinaNet, tuned (notebook 03)", "yolo_final": "YOLO11s, tuned (notebook 04)"}
others = {}
for r, label in CNN.items():
    try:
        fetch_run(r)
        others[r] = label
    except FileNotFoundError:
        print(f"{label}: no results yet ({r}), left out of the comparison")

stages = TRAINED + ["final"]
FINAL_NAME = row_name("final")
cols = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR", "ms_per_img"]
table = load_results().reindex(list(others) + [name(s) for s in stages])[cols]
table.index = list(others.values()) + [row_name(s) for s in stages]
table

**Validation mAP over training against YOLO11s under the same budget.** All curves come from Ultralytics' validation (detections in ignore regions count as false positives), so they are comparable with each other but not with the `coco_evaluate` numbers above or the curves of notebooks 02–03. The right plot shows the same budget in time.

In [ ]:
yolo_base = "yolo_baseline"
fetch_run(yolo_base)
histories = {row_name(s): json.load(open(ensure_run(s) / "history.json")) for s in TRAINED}
histories["YOLO11s baseline (notebook 04)"] = json.load(open(RUNS_DIR / yolo_base / "history.json"))
plot_history(histories)

# Convergence: epochs that fit the budget and the validation mAP gained by the last full evaluations
pd.DataFrame({k: {"epochs": len(h), "minutes": h[-1]["time_s"] / 60, "best val mAP": max(r["mAP"] for r in h),
                  "best epoch": max(h, key=lambda r: r["mAP"])["epoch"],
                  "gain in the last epoch": h[-1]["mAP"] - h[-2]["mAP"] if len(h) > 1 else float("nan"),
                  "optimizer steps": len(h) * len(subsample(load_split("train")[0])) / json.load(open(ensure_run("baseline") / "config.json"))["train"]["nbs"]}
              for k, h in histories.items() if k.startswith("RT-DETR")}).T.round(3)

**Findings.** *To fill in after the Kaggle run: RT-DETR vs the CNN detectors by size and speed, how the curves compare, and an explicit statement of whether the model converged within the budget.*

## 9. Small objects and dense scenes

**By object size:** share of GT boxes covered by a detection (IoU ≥ 0.5, any class, score ≥ 0.3), detections per image and AP per class. **By scene density:** the val frames grouped by the number of GT boxes per frame, since the scene density is what query-based matching is most sensitive to, and what NMS can hurt in the CNN detectors.

In [ ]:
# Share of GT boxes found by size and detections per image (score >= 0.3), and per-class AP
gt, det = match_errors(preds, final_cfg["image_ids"])
found, per_class = {"RT-DETR-L, tuned": size_coverage(gt)}, {"RT-DETR-L, tuned": final["per_class"]}
n_det = {"RT-DETR-L, tuned": len(det) / len(final_cfg["image_ids"])}
dup = {"RT-DETR-L, tuned": (det.outcome == "duplicate").mean() * 100}
other_preds = {}
for r, label in others.items():
    c = json.load(open(RUNS_DIR / r / "config.json"))
    other_preds[label] = json.load(open(RUNS_DIR / r / "preds_val.json"))
    g, dt = match_errors(other_preds[label], c["image_ids"])
    found[label], n_det[label], dup[label] = size_coverage(g), len(dt) / len(c["image_ids"]), (dt.outcome == "duplicate").mean() * 100
    per_class[label] = json.load(open(RUNS_DIR / r / "metrics.json"))["per_class"]
cmp = pd.DataFrame(found).rename(index=lambda b: f"GT found, {b}")
cmp.loc["detections / image"] = n_det
cmp.loc["duplicates, % of detections"] = dup
display(cmp.round(3), pd.DataFrame(per_class).T.round(3).style.set_caption("AP per class"))

In [ ]:
def by_density(results, image_ids, edges=(0, 50, 100, 10_000)):
    # COCO metrics of the frames grouped by their number of GT boxes (with the number of videos behind each group)
    frames, boxes, _ = load_split("val")
    seq = frames.set_index("image_id").seq
    n = boxes.groupby("image_id").size().reindex(image_ids, fill_value=0)
    bucket = pd.cut(n, edges, labels=["up to 50", "51-100", "over 100"], include_lowest=True)
    rows = {}
    for b in bucket.cat.categories:
        ids = bucket.index[bucket == b].tolist()
        if ids:
            m = coco_evaluate(results, "val", image_ids=ids)
            rows[b] = {"frames": len(ids), "videos": seq[ids].nunique(), "mAP": m["mAP"], "AP_small": m["AP_small"], "AR": m["AR"]}
    return pd.DataFrame(rows).T.rename_axis("GT boxes per frame")


density = {"RT-DETR-L, tuned": by_density(preds, final_cfg["image_ids"])}
if "YOLO11s, tuned (notebook 04)" in other_preds:
    density["YOLO11s, tuned"] = by_density(other_preds["YOLO11s, tuned (notebook 04)"], final_cfg["image_ids"])
pd.concat(density, axis=1).round(3)

**Findings.** *To fill in after the Kaggle run: small-object performance against YOLO11s, behaviour in the densest frames, and the duplicate rate without NMS.*

## 10. Failure cases

Errors of the tuned model at score ≥ 0.3 and IoU 0.5: outcome of every GT box by size and occlusion, class confusions and the types of false positives (same definitions as in notebook 02).

GT outcomes: **correct** = detected with the right class; **crowded** = covered only by a box already used for a neighbour; **wrong_class** = detected as another class; **mislocalized** = best IoU 0.1–0.5; **missed** = no detection. Detections: **tp** = matches a GT box; **background** = on nothing; **ignored** = centre in an ignore region; **wrong_class** = right place, wrong class; **duplicate** = second box on an already matched GT.

In [ ]:
error_tables(gt, det)
plot_confusion(gt);

Frames with the most missed small objects, the most class confusions and the most background false positives. Top: ground truth (box colour = class). Bottom: model predictions with score ≥ 0.3 (box colour = predicted class). Red areas are ignore regions.

In [ ]:
plot_failure_frames(gt, det, preds);

**Failure cases.** *To fill in after the Kaggle run: at least three failure cases with plausible causes, compared with notebook 04.*

## 11. Submission

Test predictions of the tuned model, written to `submissions/submission_<run>.csv`, validated against `test_images.csv` and uploaded to Kaggle.

In [ ]:
source = final_cfg["trained"]
if SUBMIT and ON_KAGGLE:
    kaggle_secrets()                                # fail before a long run if the credentials are missing
if RETRAIN_FULL:
    if not QUICK:
        check_hf_write()
    data = export_yolo(YOLO_DATA_DIR / ("quick" if QUICK else str(CFG["long_side"])), long_side=CFG["long_side"],
                       max_frames=CFG["train_frames"], trainval=True)
    epochs = len(json.load(open(fetch_run(source) / "history.json")))     # no val set to stop on: same epochs as the selected run
    train = {**CFG["train"], "imgsz": IMGSZ, "epochs": 1 if QUICK else epochs, "time": None, "val": False}
    train_runs([dict(name=name("full"), data=str(data), arch=CFG["arch"], weights=CFG["weights"], use_last=True,
                     **final_cfg["model"], train=train)])
    save_run(name("full"), config={**CFG, "train": train, "model": final_cfg["model"], "test": final_cfg["test"], "retrained_from": source})
    if not QUICK:
        push_run(name("full"))
    source = name("full")
if SUBMIT:
    sz = final_cfg["test"]["imgsz"]
    weights = RUNS_DIR / source / "best.pt" if RETRAIN_FULL else fetch_run(source, weights=True) / "best.pt"
    model = load_ultra(weights)
    path = make_submission(ultra_predictor(model, sz), source, long_side=frames_side(sz), label_offset=0,
                           num_workers=CFG["num_workers"])
    if not QUICK:
        submit_to_kaggle(path, f"{FINAL_NAME}{' (train + val)' if RETRAIN_FULL else ''}")
    del model
    torch.cuda.empty_cache()

## 12. Conclusion

*To fill in after the Kaggle run: final metrics, what each experiment changed, how RT-DETR compares with the CNN detectors under the same budget, convergence, and what carries over to the final pipeline (notebook 06).*